## Diffusion Map

In [ ]:
import os
from itertools import cycle

import numpy as np

import scanpy as sc
import scipy.sparse as sp

import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
def plot_gene_embedding(
    adata,
    gene: str,
    emb_key: str = "X_umap",
    threshold: float = None,
    feature_name_col: str = "feature_name",
    subclass_col: str = "subclass.l2",
    cmap: str = "Set1",
    background_color: str = "white",
    background_alpha: float = 1.0,
    point_size: float = 7.0,
    figsize: tuple = (6, 6),
    show_legend: bool = True,
    title_prefix: str = None,
    save: bool = False,
    save_dir: str = "figures/genes",
    filename: str = None,
    dpi: int = 300,
    ext: str = "png",
    bw: bool = False,
):
    """
    Plot the expression of a gene on any 2D embedding.

    Arg:
        adata (AnnData): Annotated data object containing the gene expression and embedding data.
        gene (str): The gene to plot.
        emb_key (str): Key in `adata.obsm` for the embedding (e.g. "X_umap", "X_tsne").
        threshold (float, optional): If set, shows binary above/below threshold; else continuous color.
        feature_name_col (str): Column in `adata.var` that contains the gene names.
        subclass_col (str): Column in `adata.obs` that contains the subclass labels.
        cmap (str): Color map to use for the subclasses.
        background_color (str): Color for points below the threshold.
        background_alpha (float): Alpha for the background points.
        point_size (float): Size of the points in the plot.
        figsize (tuple): Size of the figure.
        show_legend (bool): Whether to show the legend.
        title_prefix (str): Prefix for the plot title.
        save (bool): Whether to save the figure.
        save_dir (str): Directory to save the figure.
        filename (str): Filename to save the figure as. If None, defaults to a generated name.
        dpi (int): Dots per inch for the saved figure.
        ext (str): File extension for the saved figure.
        bw (bool): If True, use black and white colors for the plot.

    Returns:
        fig, ax: The figure and axis objects of the plot.
    """
    if gene not in adata.var[feature_name_col].values:
        raise ValueError(f"Gene '{gene}' not found in adata.var['{feature_name_col}'].")
    gi = np.where(adata.var[feature_name_col] == gene)[0][0]
    X = adata[:, gi].X
    expr = X.A.flatten() if sp.issparse(X) else X.flatten()

    if emb_key not in adata.obsm:
        raise KeyError(f"Embedding '{emb_key}' not found in adata.obsm.")
    coords = adata.obsm[emb_key]

    base = emb_key.replace("X_", "").upper()
    if base.startswith("PCA"):
        axis1, axis2 = "PC 1", "PC 2"
    elif base.startswith("TSNE"):
        axis1, axis2 = "TSNE 1", "TSNE 2"
    elif base.startswith("UMAP"):
        axis1, axis2 = "UMAP 1", "UMAP 2"
    elif base.lower().startswith("diffusion"):
        axis1, axis2 = "DM 1", "DM 2"
    else:
        axis1, axis2 = f"{base} 1", f"{base} 2"

    subs = list(adata.obs[subclass_col].cat.categories)
    marker_cycle = cycle(["o", "s", "^", "v", "<", ">", "P", "X"])
    marker_dict = {s: next(marker_cycle) for s in subs}

    if bw:
        gray = np.linspace(0.2, 0.8, len(subs))
        palette = [(g, g, g) for g in gray]
    else:
        try:
            palette = sns.color_palette(cmap, n_colors=len(subs))
        except ValueError:
            palette = plt.get_cmap(cmap, len(subs)).colors
    color_dict = dict(zip(subs, palette))

    marker_unicode = {
        "o": "●",
        "s": "■",
        "^": "▲",
        "v": "▼",
        "<": "◄",
        ">": "►",
        "P": "✚",
        "X": "✖",
    }

    if threshold is None:
        n_sub = len(subs)
        fig = plt.figure(figsize=(figsize[0] + 0.5 * n_sub, figsize[1]))
        gs = GridSpec(1, n_sub + 1, width_ratios=[1] + [0.1] * n_sub, wspace=0.6)
        ax = fig.add_subplot(gs[0, 0])
    else:
        fig, ax = plt.subplots(figsize=figsize)

    emb_name = base.replace("_", " ")
    ax.set_title(
        f"{title_prefix + ': ' if title_prefix else ''}{gene}"
        + (f" > {threshold}" if threshold is not None else " (continuous)")
        + f"\non {emb_name}"
    )
    ax.axis("off")

    if threshold is not None:
        bg = expr <= threshold
        ax.scatter(
            coords[bg, 0],
            coords[bg, 1],
            c=background_color,
            s=point_size,
            alpha=background_alpha,
            edgecolor="k" if bw else None,
            linewidth=0.3 if bw else 0,
            label="below threshold",
        )
        for sub in subs:
            mask = (adata.obs[subclass_col] == sub) & (expr > threshold)
            ax.scatter(
                coords[mask, 0],
                coords[mask, 1],
                c=[color_dict[sub]],
                marker=marker_dict[sub],
                s=point_size,
                edgecolor="k" if bw else None,
                linewidth=0.3 if bw else 0,
                label=f"{sub} > {threshold}",
            )
        if show_legend:
            ax.legend(
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                frameon=False,
                markerscale=1.5,
                fontsize=8,
            )
    else:
        for i, sub in enumerate(subs):
            mask = adata.obs[subclass_col] == sub
            sub_coords = coords[mask]
            expr_sub = expr[mask]
            if bw:
                cmap_sub = LinearSegmentedColormap.from_list(
                    f"{sub}_bw", ["white", "black"]
                )
            else:
                cmap_sub = LinearSegmentedColormap.from_list(
                    f"{sub}_cmap", ["white", color_dict[sub]]
                )
            sc_pts = ax.scatter(
                sub_coords[:, 0],
                sub_coords[:, 1],
                c=expr_sub,
                cmap=cmap_sub,
                s=point_size,
                alpha=1.0,
                marker=marker_dict[sub],
                edgecolor="k" if bw else None,
                linewidth=0.3 if bw else 0,
            )
            cax = fig.add_subplot(gs[0, i + 1])
            cb = fig.colorbar(sc_pts, cax=cax)
            sym = marker_unicode.get(marker_dict[sub], "")
            cb.set_label(f"{sym} Expr ({sub})", fontsize=8)
            cb.ax.yaxis.set_ticks_position("right")
            cb.ax.tick_params(axis="y", labelsize=7)

    ax.set_xlabel(axis1)
    ax.set_ylabel(axis2)

    if save:
        outdir = os.path.join(save_dir, gene)
        fn = (
            filename
            or f"{(title_prefix or gene).replace(' ','_')}_{base}"
            + (f"_{threshold}" if threshold is not None else "")
            + f".{ext}"
        )
        outdir = os.path.join(outdir, fn)
        os.makedirs(os.path.dirname(outdir), exist_ok=True)
        fig.savefig(outdir, dpi=dpi, bbox_inches="tight")
        print(f"▶ Saved {os.path.join(outdir,fn)}")

    plt.show()
    return fig, ax

In [ ]:
adata = sc.read_h5ad("data/kpmp_tal_filtered.h5ad")
adata

In [ ]:
sc.tl.diffmap(adata, n_comps=30)

In [ ]:
adata

In [ ]:
eigvals = adata.uns["diffmap_evals"]
plt.figure(figsize=(4, 3))
plt.plot(range(1, len(eigvals) + 1), eigvals, "o-")
plt.xlabel("Diffusion component")
plt.ylabel("Eigenvalue")
plt.title("Diffusion Map Scree Plot")
plt.tight_layout()
plt.show()

In [ ]:
sc.pl.diffmap(adata, components=["1,3"], color="disease")
sc.pl.diffmap(adata, components=["1,3"], color="subclass.l2")

In [ ]:
sc.pl.diffmap(
    adata, color=["nCount_RNA", "percent.mt"], cmap="viridis", components=["1,3"]
)

In [ ]:
dc = adata.obsm['X_diffmap'][:, [0, 2]]
sns.kdeplot(x=dc[:,0], y=dc[:,1], levels=5, cmap="Reds")
plt.scatter(dc[:,0], dc[:,1], s=5, c='k', alpha=0.3)
plt.show()

In [ ]:
adata.obsm['X_diffmap_1_3'] = adata.obsm['X_diffmap'][:, [0, 2]]

fig, ax = plot_gene_embedding(
    adata,
    gene="UMOD",
    emb_key="X_diffmap_1_3",
    cmap="magma",
    point_size=5.0,
    title_prefix="DiffMap DC1 vs DC3"
)